In [1]:
import numpy as np
import pandas as pd

In [2]:
tuning_all = pd.read_csv('tuning-results.csv')
# tuning_binary = pd.read_csv("tuning-results-binary.csv")

In [3]:
best_lambdas_all = {}
for dataset in tuning_all['data_name'].unique():
    for prerank in tuning_all.prerank.unique():
        subdf = tuning_all[(tuning_all['data_name'] == dataset) & (tuning_all['prerank'] == prerank)]
        if len(subdf['lambda'].values) == 6:
            if 0.0 not in subdf['lambda'].values:
                continue
            baseline_energy = subdf[subdf['lambda'] == 0.0]['energy'].values[0]
            valid = subdf[subdf['energy'] <= 1.1 * baseline_energy]
            best = valid.sort_values('pce').iloc[0] if not valid.empty else subdf.sort_values('pce').iloc[0]
            best_lambdas_all[(dataset, prerank)] = best['lambda'] 
        else: print(f"not enough lambdas for {dataset} {prerank}")

not enough lambdas for births1 dependency
not enough lambdas for wage dependency
not enough lambdas for meps_21 dependency
not enough lambdas for meps_19 dependency
not enough lambdas for meps_20 dependency
not enough lambdas for house dependency
not enough lambdas for bio dependency
not enough lambdas for blog_data dependency
not enough lambdas for calcofi dependency
not enough lambdas for ansur2 dependency
not enough lambdas for taxi marginal
not enough lambdas for taxi mean
not enough lambdas for taxi variance
not enough lambdas for taxi dependency
not enough lambdas for taxi pca
not enough lambdas for taxi density
not enough lambdas for taxi cdf


In [ ]:
best_lambdas_all

In [5]:
best_lambdas_all[('calcofi', 'cdf')]

np.float64(5.0)

In [2]:
df = pd.read_csv('metrics-after-reg-ansur-taxi.csv')
df.head()

,data_name,seed,prerank,pce,nll,energy,mse
0,households,0,marginal,0.028741,3.295859,0.951055,0.663382
1,households,42,marginal,0.031749,3.270183,0.882693,0.554523
2,households,866,marginal,0.029671,3.396923,0.965792,0.652964
3,households,12,marginal,0.029982,3.423873,0.960883,0.667672
4,households,4,marginal,0.027437,3.255520,0.935410,0.623558


In [3]:
df.shape

(665, 7)

In [4]:
agg_df = (
     df.groupby(["data_name", "prerank"])
      .agg(pce_mean=("pce", "mean"),
           pce_se=("pce", lambda x: x.std() / (len(x) ** 0.5)),
           nll_mean=("nll", "mean"),
           nll_se=("nll", lambda x: x.std() / (len(x) ** 0.5)),
           energy_mean=("energy", "mean"),
           energy_se=("energy", lambda x: x.std() / (len(x) ** 0.5)),
           mse_mean=("mse", "mean"),
           mse_se=("mse", lambda x: x.std() / (len(x) ** 0.5)))
          ).reset_index()

In [5]:
agg_df[agg_df['data_name']=='calcofi']

,data_name,prerank,pce_mean,pce_se,nll_mean,nll_se,energy_mean,energy_se,mse_mean,mse_se
42,calcofi,cdf,0.020244,0.000079,0.592544,0.001261,0.421565,0.000632,0.343321,0.003156
43,calcofi,density,0.020344,0.000137,0.600525,0.002550,0.420803,0.000437,0.331909,0.000906
44,calcofi,dependency,0.019735,0.000173,0.595023,0.004594,0.420619,0.000445,0.332179,0.001040
45,calcofi,marginal,0.019996,0.000149,0.593159,0.003412,0.420918,0.000647,0.334658,0.001288
46,calcofi,mean,0.021121,0.000182,0.585928,0.002176,0.420979,0.000487,0.336832,0.001716
47,calcofi,pca,0.020883,0.000169,0.597101,0.003944,0.420865,0.000462,0.336415,0.000520
48,calcofi,variance,0.019913,0.000222,0.598236,0.005161,0.420815,0.000439,0.333645,0.001294


In [ ]:
dataset = "calcofi"
subdf = agg_df[agg_df["data_name"] == dataset]
# for _, row in subdf.iterrows():
#     print(f"""{row['prerank']}: {row['nll_mean']:.3f} ({row['nll_se']:.3f}) & {row['energy_mean']:.3f} ({row['energy_se']:.3f}) & {row['mse_mean']:.3f} ({row['mse_se']:.3f})""")

,data_name,prerank,pce_mean,pce_se,nll_mean,nll_se,energy_mean,energy_se,mse_mean,mse_se
13,births1,cdf,0.026634,0.001301,1.391951,0.077995,0.729287,0.004198,0.860039,0.009327
14,births1,density,0.034024,0.003453,2.120752,0.041119,0.711908,0.003201,0.840337,0.006422
15,births1,dependency,0.027086,0.001969,1.629785,0.119460,0.714112,0.006299,0.852112,0.011010
16,births1,marginal,0.028327,0.000917,1.820428,0.174687,0.724457,0.005878,0.852920,0.008848
17,births1,mean,0.026544,0.001660,1.372964,0.083237,0.733607,0.007451,0.873195,0.015179
18,births1,pca,0.034281,0.000435,1.363047,0.092498,0.729912,0.004354,0.868853,0.011757
19,births1,variance,0.031463,0.002083,1.738637,0.196139,0.718074,0.005310,0.858125,0.013846


In [6]:
order = ["marginal", "mean", "variance", "dependency", "pca", "density", "cdf"]
subset = agg_df[agg_df["data_name"] == "calcofi"]
subset = subset.set_index("prerank").loc[order]
result = " & ".join(f"{row['pce_mean']:.3f} ({row['pce_se']:.3f})" for _, row in subset.iterrows())
print(result)


0.020 (0.000) & 0.021 (0.000) & 0.020 (0.000) & 0.020 (0.000) & 0.021 (0.000) & 0.020 (0.000) & 0.020 (0.000)


#### Comapring marginal+prerank versus PCA+prerank

In [16]:
marg_prerank_df = pd.read_csv("metrics-after-reg-double-reg2.csv")
pca_prerank_df = pd.read_csv("metrics-after-reg-double-reg3.csv")

In [17]:
marg_prerank_df = marg_prerank_df[marg_prerank_df['data_name'].isin(['scm20d', 'scm1d'])]

In [21]:
agg_pca_df = (
     pca_prerank_df.groupby(["data_name", "prerank"])
      .agg(pce_marg_mean=("pce_marg", "mean"),
           pce_marg_se=("pce_marg", lambda x: x.std() / (len(x) ** 0.5)),
           pce_prerank_mean=("pce_prerank", "mean"),
           pce_prerank_se=("pce_prerank", lambda x: x.std() / (len(x) ** 0.5))
          ).reset_index())

In [22]:
agg_pca_df.head(10)

,data_name,prerank,pce_marg_mean,pce_marg_se,pce_prerank_mean,pce_prerank_se
0,scm1d,cdf,0.036008,0.008737,0.042283,0.009344
1,scm1d,density,0.043398,0.003746,0.095975,0.001408
2,scm1d,dependency,0.030439,0.001933,0.021307,0.002132
3,scm1d,mean,0.030188,0.000517,0.022808,0.002331
4,scm1d,variance,0.031830,0.002824,0.052335,0.005751
5,scm20d,cdf,0.032419,0.001063,0.029351,0.001417
6,scm20d,density,0.049071,0.003453,0.087985,0.003604
7,scm20d,dependency,0.044501,0.001533,0.025077,0.001354
8,scm20d,mean,0.033802,0.001598,0.021937,0.001135
9,scm20d,variance,0.034356,0.001393,0.044767,0.002594


In [23]:
order = ["mean", "variance", "dependency", "density", "cdf"]
subset = agg_pca_df[agg_pca_df["data_name"] == "scm20d"]
subset = subset.set_index("prerank").loc[order]
for _, row in subset.iterrows():
    print(f"{row['pce_marg_mean']:.3f} ({row['pce_marg_se']:.3f}) & {row['pce_prerank_mean']:.3f} ({row['pce_prerank_se']:.3f})")

0.034 (0.002) & 0.022 (0.001)
0.034 (0.001) & 0.045 (0.003)
0.045 (0.002) & 0.025 (0.001)
0.049 (0.003) & 0.088 (0.004)
0.032 (0.001) & 0.029 (0.001)
